In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURACIÓN DE ARCHIVOS
# ==========================================
archivo_datos_reales = "data/neuronas_120.h5"
archivo_datos_generados = "sec_test.h5"
train_size = 0.6  # El porcentaje que usaste para entrenar

# ==========================================
# 2. CARGA DE DATOS
# ==========================================
# Cargar datos reales y quedarnos solo con el TEST set
with h5py.File(archivo_datos_reales, "r") as f:
    datos_reales = f["samples"][()]
    
split_idx = int(train_size * datos_reales.shape[0])
test_reales = datos_reales[split_idx:]  # Forma: (Tiempo_Test, 120)

# Cargar datos generados
with h5py.File(archivo_datos_generados, "r") as f:
    datos_gen = f["generated_sequences"][()]
    
# Si generaste múltiples secuencias, las aplanamos todas en una única dimensión temporal
# Forma original: (Num_Seqs, Tiempo_Gen, 120) -> Forma final: (Num_Seqs * Tiempo_Gen, 120)
test_gen = datos_gen.reshape(-1, datos_gen.shape[-1])

# ==========================================
# 3. CÁLCULO DE MÉTRICAS
# ==========================================
print(f"Calculando métricas...\nDatos reales (Test): {test_reales.shape}\nDatos generados: {test_gen.shape}")

# Firing Rate: Media de activación de cada neurona (eje 0 = a lo largo del tiempo)
fr_real = test_reales.mean(axis=0)
fr_gen = test_gen.mean(axis=0)

# Pairwise Correlations: Matriz de correlación de Pearson
# np.corrcoef espera que las variables sean filas, por eso transponemos (.T)
corr_real_matrix = np.corrcoef(test_reales.T)
corr_gen_matrix = np.corrcoef(test_gen.T)

# Si una neurona está "muerta" (siempre 0), corrcoef devuelve NaN. Lo rellenamos con 0.
corr_real_matrix = np.nan_to_num(corr_real_matrix)
corr_gen_matrix = np.nan_to_num(corr_gen_matrix)

# Extraemos solo el triángulo superior de la matriz para no tener pares duplicados (i,j) y (j,i)
indices_triangulo = np.triu_indices_from(corr_real_matrix, k=1)
corr_real = corr_real_matrix[indices_triangulo]
corr_gen = corr_gen_matrix[indices_triangulo]

# ==========================================
# 4. VISUALIZACIÓN (PLOTS)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ---- PLOT 1: Firing Rates ----
axes[0].scatter(fr_real, fr_gen, alpha=0.7, color="royalblue", edgecolors="k")
# Dibujar línea ideal y=x
limites_fr = [min(fr_real.min(), fr_gen.min()), max(fr_real.max(), fr_gen.max())]
axes[0].plot(limites_fr, limites_fr, "r--", label="Ideal (y=x)")

axes[0].set_title("Firing Rates (120 Neuronas)", fontsize=14)
axes[0].set_xlabel("Test Real", fontsize=12)
axes[0].set_ylabel("CRBM Generado", fontsize=12)
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.5)

# ---- PLOT 2: Pairwise Correlations ----
axes[1].scatter(corr_real, corr_gen, alpha=0.3, color="seagreen")
# Dibujar línea ideal y=x
limites_corr = [min(corr_real.min(), corr_gen.min()), max(corr_real.max(), corr_gen.max())]
axes[1].plot(limites_corr, limites_corr, "r--", label="Ideal (y=x)")

axes[1].set_title("Pairwise Correlations", fontsize=14)
axes[1].set_xlabel("Test Real", fontsize=12)
axes[1].set_ylabel("CRBM Generado", fontsize=12)
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()